## Problem Statement

In the highly competitive taxi booking industry, maximizing driver revenue is essential for operational sustainability and customer satisfaction. Taxi companies process millions of rides daily, generating large volumes of transactional data that can reveal important behavioral patterns.

One critical business question is whether a passenger’s payment method influences the fare amount and overall revenue generated per trip. Understanding this relationship can help taxi companies optimize payment strategies, improve driver incentives, and design targeted promotional campaigns.

This project uses statistical hypothesis testing and exploratory data analysis to investigate whether there is a statistically significant difference in fare amounts between customers who pay using credit cards and those who pay using cash.

Using Python, data cleaning, visualization, and inferential statistics, this project transforms raw taxi trip records into actionable business insights.

## Research Question
### Primary Research Question

Does the passenger payment method significantly affect taxi fare amount?

### Statistical Research Question

**Null Hypothesis (H₀):**
There is no significant difference in average fare amount between payment methods. This means the average (mean) values for credit card and cash payments are equal.

$$
H_0: \mu_{\text{credit card}} = \mu_{\text{cash}}
$$


**Alternative Hypothesis (H₁):**

$$
H_1: \mu_{\text{credit card}} \ne \mu_{\text{cash}}
$$

This means there is a statistically significant difference in the average fare amount between credit card and cash payment methods.

## Business Objective
- Identify which payment method generates higher average revenue
- Help taxi companies optimize payment strategies
- Provide data-driven recommendations for driver earnings improvement
- Demonstrate the practical use of A/B testing in transportation analytics

## Project Agenda
1. Business Understanding
- Understand the taxi revenue problem
- Define research objectives
2. Data Collection
- Import NYC taxi trip dataset
3. Data Cleaning
- Remove irrelevant columns
- Handle missing values
- Remove outliers
- Filter invalid fares
4. Exploratory Data Analysis (EDA)
- Analyze payment type distribution
- Compare fare distributions
- Visualize customer behavior
5. Statistical Hypothesis Testing
- Perform independent two-sample t-test
- Evaluate p-value
- Interpret statistical significance
6. Business Insights
- Identify high-value payment methods
- Revenue optimization recommendations
7. Conclusion
- Summarize findings
- Discuss limitations and future improvements

## Import Libraries

In [24]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from scipy.stats import ttest_ind

## Load Dataset

In [3]:
# Get list of all parquet files
parquet_files = glob.glob('*.parquet')
# Read all into a list, then concat
df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)

In [4]:
df.shape

(11077206, 20)

In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


## Data Preparing and Cleaning

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11077206 entries, 0 to 11077205
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee         

In [8]:
df['duration' ] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
df['duration'] = df['duration'].dt.total_seconds()/60
df.head(2)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,duration
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,5.550000
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75,5.716667


In [9]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'duration'],
      dtype='object')

In [25]:
required_columns = ['payment_type','passenger_count', 'trip_distance', 'fare_amount','tip_amount', 'total_amount','duration']
data = df[required_columns]
data.head()

,payment_type,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,duration
0,1,1.0,0.97,7.2,3.66,15.86,5.550000
1,2,0.0,0.90,7.9,0.00,13.65,5.716667
2,1,0.0,1.40,10.7,2.50,18.95,8.883333
3,1,4.0,5.58,38.7,11.11,55.56,42.800000
4,1,0.0,2.16,13.5,3.85,23.10,13.500000


#### Handle Missing Values

In [26]:
data.isnull().sum()

payment_type             0
passenger_count    3057123
trip_distance            0
fare_amount              0
tip_amount               0
total_amount             0
duration                 0
dtype: int64

In [27]:
data.isnull().sum()/len(df)

payment_type       0.000000
passenger_count    0.275983
trip_distance      0.000000
fare_amount        0.000000
tip_amount         0.000000
total_amount       0.000000
duration           0.000000
dtype: float64

In [29]:
data.dropna(inplace = True)
data.shape

(8020083, 7)

In [32]:
data.head()

,payment_type,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,duration
0,1,1.0,0.97,7.2,3.66,15.86,5.550000
1,2,0.0,0.90,7.9,0.00,13.65,5.716667
2,1,0.0,1.40,10.7,2.50,18.95,8.883333
3,1,4.0,5.58,38.7,11.11,55.56,42.800000
4,1,0.0,2.16,13.5,3.85,23.10,13.500000


In [35]:
data['passenger_count'] = data['passenger_count'].astype('int16')
data.dtypes

payment_type         int64
passenger_count      int16
trip_distance      float64
fare_amount        float64
tip_amount         float64
total_amount       float64
duration           float64
dtype: object

In [37]:
data[data.duplicated()]

,payment_type,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,duration
975,1,1,1.58,12.8,3.71,22.26,12.716667
2301,1,1,0.80,6.5,2.30,13.80,4.700000
2315,1,1,0.56,5.8,2.31,13.86,3.533333
3717,1,1,0.66,8.6,2.87,17.22,7.833333
5080,1,1,1.04,7.9,2.73,16.38,5.633333
...,...,...,...,...,...,...,...
10131435,1,1,1.12,7.9,2.73,16.38,5.950000
10131447,1,1,0.84,8.6,2.87,17.22,7.450000
10131448,1,1,3.15,16.3,4.41,26.46,13.100000
10131454,1,1,1.90,11.4,3.43,20.58,9.600000


In [39]:
data.drop_duplicates(inplace= True)
data.shape

(6573185, 7)

#### Understand Payment Type Encoding

Usually:

1 = Credit Card; </br>
2 = Cash

In [40]:
data['payment_type'].value_counts(normalize= True)

payment_type
1    0.853670
2    0.122815
4    0.018127
3    0.005387
Name: proportion, dtype: float64

In [43]:
# Keep Only Credit Card and Cash Payments
data = data[data['payment_type'].isin([1,2])]
data.shape

(6418619, 7)

## Exploratory Data Analysis